In [113]:
using HDF5
using Unitful
import PhysicalConstants.CODATA2018: m_n, e, ħ
using FileIO
using GeometryBasics
using LinearAlgebra
using BenchmarkTools
using MeshIO
using SparseArrays
using StaticArrays

In [112]:
# Defining a function to calculate the magnitude of the wavevector, in Angstrom^-1, of the neutron from its energy.

"""
Calculates the magnitude of the wavevector, in Angstrom^-1, from an energy, in meV.

Parameters
----------
en (float): Energy, in meV.

Returns
-------
mag_k (float): Magnitude of wavevector, in Angstrom^-1.
"""
function k_calc(en :: AbstractFloat)
    mag_k = sqrt(2 * m_n * en * e * (1e-3)) / (ħ * (1e10))
    return ustrip(mag_k)
end

k_calc

In [111]:
# Extracting the contents of the .nxspe file.

en_i, azi, pol, data, del_en = h5open("test_nxspe_data/LET104215_3.7meV_1to1.nxspe", "r") do f
    # Initial energy in meV.
    en_i = read(f["ws_out/NXSPE_info/fixed_energy"])[1]
    # Azimuthal angles in degrees.
    azi = read(f["ws_out/data/azimuthal"])
    # Polar angles in degrees.
    pol = read(f["ws_out/data/polar"])
    # Measured signal for each energy bin and each detector.
    data = read(f["ws_out/data/data"])
    # Energy bin centres, in meV, equal to energy change of neutron.
    del_en = read(f["ws_out/data/energy"])
    return en_i, azi, pol, data, del_en
end

(3.7, [-137.16071701049805, -137.39026260375977, -137.62151336669922, -137.8544692993164, -138.08910751342773, -138.32544326782227, -138.56351470947266, -138.80327224731445, -139.0447883605957, -139.2881088256836  …  41.245439529418945, 41.48468208312988, 41.7221097946167, 41.9577579498291, 42.19169521331787, 42.42388725280762, 42.65436553955078, 42.883137702941895, 43.110212326049805, 43.33559226989746], [48.28250598907471, 48.177175521850586, 48.07186985015869, 47.9666051864624, 47.8613977432251, 47.75627422332764, 47.651217460632324, 47.546268463134766, 47.44140434265137, 47.33662223815918  …  131.6914520263672, 131.5868377685547, 131.4821891784668, 131.37750625610352, 131.27277374267578, 131.16801834106445, 131.06324005126953, 130.95844650268555, 130.8536720275879, 130.74888229370117], [NaN NaN … NaN NaN; NaN NaN … NaN NaN; … ; NaN NaN … NaN NaN; NaN NaN … NaN NaN], [-2.9600000000000004, -2.9415000000000004, -2.9230000000000005, -2.9045000000000005, -2.8860000000000006, -2.86750000

In [5]:
# Calculating en_f, ki, n_bins, n_detectors from the extracted content of the .nxspe file.


# Finding the initial wavevector, in Angstrom^-1, from the initial energy.
ki = zeros(3)
ki[1] = k_calc(en_i)
# Converting ki to a static array.
ki = SVector{3}(ki)

# Extracting the number of energy bins and number of detectors.
const n_bins = length(del_en) - 1
const n_detectors = length(azi)

# Finding the bin centres by averaging the energies on each end of the bin.
en_bins = zeros(n_bins)
for i in 1:n_bins
    en_bins[i] = (del_en[i] + del_en[i+1]) / 2
end

# Determining the final neutron energy, en_f, based on which energy bin we are considering.
ei_bins = en_i * ones(n_bins)
ef_bins = ei_bins - en_bins

# Converting the bins to static arrays.
ef_bins = SVector{n_bins}(ef_bins)

320-element SVector{320, Float64} with indices SOneTo(320):
 6.65075
 6.632250000000001
 6.6137500000000005
 6.595250000000001
 6.5767500000000005
 6.558250000000001
 6.539750000000001
 6.521250000000001
 6.502750000000001
 6.484250000000001
 ⋮
 0.8972500000000094
 0.8787500000000099
 0.8602500000000095
 0.84175000000001
 0.8232500000000096
 0.8047500000000101
 0.7862500000000097
 0.7677500000000101
 0.7492500000000049

In [6]:
# Calculating the final wavevector, in Angstrom^-1, for each energy bin and each detector.


mag_kf = k_calc.(ef_bins)
# Reshaping the arrays to allow for broadcasting.
pol_col = reshape(pol, :, 1)
azi_col = reshape(azi, :, 1)
mag_kf_row = reshape(mag_kf, 1, :)
# Determing the components of the final wavevector using broadcasting
kx = mag_kf_row .* (sin.(deg2rad.(pol_col)) .* cos.(deg2rad.(azi_col)))
ky = mag_kf_row .* (sin.(deg2rad.(pol_col)) .* sin.(deg2rad.(azi_col)))
kz = mag_kf_row .* cos.(deg2rad.(pol_col))
# Reshaping these final wavevector grids to align with the grid of data.
kx = transpose(kx)
ky = transpose(ky)
kz = transpose(kz)

320×98304 transpose(::Matrix{Float64}) with eltype Float64:
 1.1922    1.19465   1.19711   1.19955   …  -1.17438   -1.1719    -1.16942
 1.19054   1.19299   1.19544   1.19788      -1.17274   -1.17027   -1.16779
 1.18888   1.19133   1.19377   1.19621      -1.17111   -1.16864   -1.16616
 1.18721   1.18966   1.1921    1.19454      -1.16947   -1.167     -1.16453
 1.18555   1.18799   1.19043   1.19286      -1.16783   -1.16536   -1.1629
 1.18388   1.18632   1.18875   1.19118   …  -1.16618   -1.16372   -1.16126
 1.18221   1.18464   1.18707   1.1895       -1.16454   -1.16208   -1.15962
 1.18053   1.18297   1.18539   1.18782      -1.16289   -1.16044   -1.15798
 1.17886   1.18129   1.18371   1.18613      -1.16124   -1.15879   -1.15634
 1.17718   1.17961   1.18203   1.18444      -1.15958   -1.15714   -1.15469
 ⋮                                       ⋱                        
 0.437895  0.438797  0.439697  0.440596     -0.431349  -0.43044   -0.429529
 0.433357  0.43425   0.435141  0.43603      -0.4

In [7]:
# Retrieving the vertices and indices of the triangular mesh of the sample surface from the .stl file.


stl = load("crystal.stl")
vertices = GeometryBasics.coordinates(stl)
indices = GeometryBasics.faces(stl)
# Extracting the number of triangular faces used in the mesh.
const n_faces = length(indices)

2142

In [8]:
# Calculating and storing the vectors parallel to each face, e2 = V2 - V1 and e3 = V3 - V1, in preparation for hte Moller-Trumbore Algorithm.
# The vertices of the triangular faces are labelled V1, V2, V3.

e2s = vertices[getindex.(indices, 2)] - vertices[getindex.(indices, 1)]
e3s = vertices[getindex.(indices, 3)] - vertices[getindex.(indices, 1)]

2142-element Vector{Point{3, Float32}}:
 [0.20006466, -0.12500381, 0.021842957]
 [-0.086564064, -0.24712181, 0.03823471]
 [-0.13139248, 0.22102165, -0.017211914]
 [0.11132717, -0.22167778, 0.014579773]
 [-0.1242342, -0.2677498, 0.004627228]
 [0.13139248, -0.22102165, 0.017211914]
 [-0.22294426, 0.20596504, 0.0055160522]
 [0.24161053, 0.0130290985, 0.010730743]
 [-0.16730404, 0.17181778, 0.01745224]
 [-0.32619762, -0.06916809, -0.0008392334]
 ⋮
 [0.28863525, -0.07749939, -0.0007972717]
 [-0.2492981, 0.23089218, -0.0039901733]
 [0.11425209, -0.24721527, 0.021465302]
 [-0.31781864, 0.21372223, -0.01871872]
 [0.26812553, -0.19742012, 0.006767273]
 [-0.03742695, 0.2215786, -0.019702911]
 [-0.28863525, 0.07749939, 0.0007972717]
 [-0.26812553, 0.19742012, -0.006767273]
 [0.050290108, 0.3967991, -0.016407013]

In [9]:
# Determining the coordinates of the points within our sample used for the Monte Carlo approximation of the volume integral.

const n_mc = 1
mc_coords = zeros(n_mc, 3)
# For this test, only one sample point used (at midpoint of coordinates).
max_coord = [maximum(getindex.(vertices, 1)), maximum(getindex.(vertices, 2)), maximum(getindex.(vertices, 3))]
min_coord = [minimum(getindex.(vertices, 1)), minimum(getindex.(vertices, 2)), minimum(getindex.(vertices, 3))]
mc_coords[1, :] = (min_coord + max_coord) / 2

3-element Vector{Float32}:
 16.6475
 20.065285
 47.947563

In [110]:
# Setting the (estimated) parameters of the sample.

# The number density of the sample in cm^-3.
const n = 1e23
# The reference absorption cross section at 25.3 meV in cm^2.
const axs_ref = 1e-23
const en_ref = 25.3

25.3

In [109]:
# Defining the function that calculates the absorption cross sections for the inputted energy.

"""
Determines the absorption cross section (axs) for the inputted energy based on the absorption of the sample at a known, reference energy.

Parameters
----------
en (float): Energy in meV.
axs_ref (float): Absorption cross section at en_ref in cm^2.
en_ref (float): Reference energy in meV.

Returns
-------
axs (float): Absorption cross section in cm^2.
"""
function axs_calc(
    en :: Float64, 
    axs_ref :: Float64, 
    en_ref :: Float64
)
    return axs_ref * sqrt(en_ref / en)
end
@benchmark axs_calc(en_i, axs_ref, en_ref)

BenchmarkTools.Trial: 10000 samples with 1000 evaluations per sample.
 Range (min … max):  4.000 ns … 68.200 ns  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     4.500 ns              ┊ GC (median):    0.00%
 Time  (mean ± σ):   5.750 ns ±  2.514 ns  ┊ GC (mean ± σ):  0.00% ± 0.00%

  █▄▂                              █                          
  ████▄▃▄▃▃▃▃▅▃▄▂▂▂▁▂▂▂▁▁▂▂▁▂▁▂▁▁▁▂█▁▂▁▂▂▂▂▂▁▂▂▁▂▂▂▂▂▂▂▂▂▂▁▂ ▃
  4 ns           Histogram: frequency by time        13.3 ns <

 Memory estimate: 0 bytes, allocs estimate: 0.

In [108]:
# Calculating the pre-scattering absorption cross section.

const axsi = axs_calc(en_i, axs_ref, en_ref)

2.614925971770107e-23

In [ ]:
# Creating the function to determine the length of the paths the neutrons take within the sample.
# Not broadcasted.

"""
Calculates the distance between a given point in the sample (origin) and a triangular face of the mesh that describes the surface. 
Iterates through each face to determine which one is intersected.
Exploits the method described in 'Fast, Minimum Storage Ray-Triangle Intersection' by Moller and Trumbore.

Parameters
----------
e2s (n_faces-vector of 3-vectors with float elements): Array containing vectors parallel to each face, equal to V2 - V1.
e3s (n_faces-vector of 3-vectors with float elements): Array containing vectors parallel to each face, equal to V3 - V1.
d (3-vector with float elements): Direction vector.
ps (n_faces-vector of 3-vectors with float elements): Array containing p = d x e3 for each face.
dets (n_faces-vector with float elements): Array containing det = p.e2 = (d x e3).e2 for each face.
origin (3-vector with float elements): Coordinates of scattering sites.
vertices (n_faces-vector of 3-vectors with float elements): Vertices of triangular faces.
indices (n_faces-vector of 3-vectors with integer elements): Indices describing which vertices form which triangles.

Returns
-------
path_length (float): Distance between origin and the surface the neutron path intersects, in units of the .stl file.
"""
function len_calc(
    e2s :: AbstractVector{<:Point{3}}, 
    e3s :: AbstractVector{<:Point{3}}, 
    d :: AbstractVector{Float64}, 
    ps :: AbstractVector{<:AbstractVector{Float64}}, 
    dets :: AbstractVector{Float64}, 
    origin :: AbstractVector{Float64}, 
    vertices :: AbstractVector{<:Point{3}}, 
    indices :: AbstractVector
)
    # If det = p.e2 = (d x e3).e2 = 0, the path is parallel to the triangular face, so it can never intersect it.
    # Keeping only positive determinants, equivalent to triangular faces in the forward direction.
    idx = findall(dets .> 1e-10)
    # Pre-calculating the inverse determinant for each face.
    inv_det = 1 ./ dets
    # Iterating through each valid face.
    for j in idx
        # Calculating t = origin - V1 and q = t x e2 required for the MT algorithm.
        t = origin - vertices[indices[j][1]]
        q = cross(t, e2s[j])
        # Calculating the barycentric coordinates, (u,v), of the intersection.
        u = inv_det[j] * (dot(ps[j], t))
        v = inv_det[j] * (dot(q, d))
        # Determining whether the intersection point lies within the triangle.
        if v ≥ 0 && u ≥ 0 && (u + v) ≤ 1
            # Neutron's path dot(q, e3s[j])
            λ = inv_det[j] * dot(q, e3s[j])
            # Determining the path length based on λ and the magnitude of the inputted direction vector.
            path_length = abs(λ) * norm(d)
            return path_length
        end
    end
end
@benchmark len_calc(e2s, e3s, -ki, cross.(fill(-ki, n_faces), e3s), dot.(p_i, e2s), mc_coords[1,:], vertices, indices)

BenchmarkTools.Trial: 10000 samples with 1 evaluation per sample.
 Range (min … max):   18.200 μs … 243.925 ms  ┊ GC (min … max):  0.00% … 99.94%
 Time  (median):      93.700 μs               ┊ GC (median):     0.00%
 Time  (mean ± σ):   121.358 μs ±   2.624 ms  ┊ GC (mean ± σ):  28.07% ±  1.41%

  ▇▄▅▄▃▁                 ▅▆▆▇███▆▅▄▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁  ▁▁▁         ▃
  ██████▇▇▅▇▇▇▆▅▅▅▅▅▅▄▅▄▇████████████████████████████████▇▇▇▇▇▆ █
  18.2 μs       Histogram: log(frequency) by time        183 μs <

 Memory estimate: 143.97 KiB, allocs estimate: 26.

In [36]:
# The pre-scattering neutron path lengths are dependent only on the MC coordinates.
# They can, therefore, be calculated and stored.

len_i = zeros(n_mc)
p_i = cross.(fill(-ki, n_faces), e3s)
det_i = dot.(p_i, e2s)
for i in 1:n_mc
    len_i[i] = len_calc(e2s, e3s, -ki, p_i, det_i, mc_coords[i, :], vertices, indices)
end

In [ ]:
# Defining the function that will calculate the attenuation factor given a certain energy bin and wavevector.

"""
Calculates the attenuation factor given a set initial and final energy and wavevector.

Parameters
----------
ki (3-vector with float elements): Pre-scattering wavevector of neutron, in Angstrom^-1.
kf (3-vector with float elements): Post-scattering wavevector of neutron, in Angstrom^-1.
en_f (float): Post-scattering energy of neutron, in meV.
vertices (n_faces-vector of 3-vectors with float elements): Vertices of the triangular faces.
indices (n_faces-vector of 3-vectors with integer elements): Indices describing which vertices correspond to which triangles.
e2s (n_faces-vector of 3-vectors with float elements): Array containing vectors parallel to each face, equal to V2 - V1.
e3s (n_faces-vector of 3-vectors with float elements): Array containing vectors parallel to each face, equal to V3 - V1.
mc_coords (n_mc-vector of 3-vectors with float elements): Coordinates of sample points used in MC method.
len_i (n_mc-vector of float elements): Pre-scattering path length of neutron, in units of .stl file.

Returns
-------
atten_calc (float): Attenuation factor.
"""
function atten_calc(
    ki :: SVector{3, Float64}, 
    kf :: SVector{3, Float64}, 
    en_f :: Float64, 
    vertices :: AbstractVector{<:Point{3}}, 
    indices :: AbstractVector, 
    e2s :: AbstractVector{<:Point{3}}, 
    e3s :: AbstractVector{<:Point{3}}, 
    mc_coords :: AbstractMatrix{Float64}, 
    len_i :: AbstractVector{Float64}
)
    # Calculating the absorption cross section after the neutron scatters.
    axsf = axs_calc(en_f, axs_ref, en_ref)
    # Setting up the Moller-Trumbore algorithm for ray-triangle intersections.
    # Calculating cross products, p = d x e3, for the direction vector, df = kf, and for each face.
    dfs = [kf for face in 1:n_faces]
    p_f = cross.(dfs, e3s)
    # Calculating the determinant = p.e2 = (d x e3).e2 for the direction vector, df, and for each face.
    det_f = dot.(p_f, e2s)
    atten_calc = 0
    for i in 1:n_mc
        # Calculating the path length, len_f, at this sample point.
        len_f = len_calc(e2s, e3s, kf, p_f, det_f, mc_coords[i, :], vertices, indices)
        # Adding the attenuation factor contribution from this sample point to A.
        atten_calc += (1 / n_mc) * exp(-n * axsi * len_i[i]) * exp(-n * axsf * len_f)
    end
    return atten_calc
end
@benchmark atten_calc(ki, SVector{3}([kx[1,1], ky[1,1], kz[1,1]]), ef_bins[1], vertices, indices, e2s, e3s, mc_coords, len_i)

BenchmarkTools.Trial: 10000 samples with 1 evaluation per sample.
 Range (min … max):   15.800 μs … 168.419 ms  ┊ GC (min … max):  0.00% … 99.90%
 Time  (median):      91.200 μs               ┊ GC (median):     0.00%
 Time  (mean ± σ):   111.461 μs ±   2.004 ms  ┊ GC (mean ± σ):  24.85% ±  1.41%

    ▁                      ▁▅█▇▁                                 
  ▂▇█▇▅▄▃▂▂▂▂▂▂▂▂▂▂▂▂▁▂▁▂▅▆█████▆▅▄▄▃▃▃▃▃▃▃▂▂▂▂▂▃▂▂▂▂▂▃▂▂▂▂▂▂▂▂ ▃
  15.8 μs          Histogram: frequency by time          184 μs <

 Memory estimate: 143.57 KiB, allocs estimate: 28.

In [ ]:
# Defining the function that calculates the grid of attenuation factors.
# Dense arrays.

"""
Calculates the attenuation factor for every non-zero, non-NaN, signal and stores in a grid of detector against energy bin.

Parameters
----------
data (n_bins x n_detectors matrix with float elements): Neutron signal measured at different detectors for different energy bins.
kx (n_bins x n_detectors matrix with float elements): Post-scattering neutron wavevector component in x direction, in Angstrom^-1.
ky (n_bins x n_detectors matrix with float elements): Post-scattering neutron wavevector component in y direction, in Angstrom^-1.
kz (n_bins x n_detectors matrix with float elements): Post-scattering neutron wavevector component in z direction, in Angstrom^-1.
ki (3-vector with float elements): Pre-scattering neutron wavevector, in Angstrom^-1.
en_i (float): Pre-scattering neutron energy, in meV.
ef_bins (n_bins-vector with float elements): Post-scattering neutron energy of each bin, in meV.
vertices (n_faces-vector of 3-vectors with float elements): Vertices of the triangular faces.
indices (n_faces-vector of 3-vectors with integer elements): Indices describing which vertices correspond to which triangles.
e2s (n_faces-vector of 3-vectors with float elements): Array containing vectors parallel to each face, equal to V2 - V1.
e3s (n_faces-vector of 3-vectors with float elements): Array containing vectors parallel to each face, equal to V3 - V1.
mc_coords (n_mc-vector of 3-vectors with float elements): Coordinates of sample points used in MC method.
len_i (n_mc-vector with float elements): Pre-scattering path length of neutron, in units of .stl file.

Returns
-------
A_grid (n_bins x n_detectors matrix of floats): Attenuation factor grid.
"""
function a_grid_calc(
    data :: AbstractMatrix{<:AbstractFloat}, 
    kx :: AbstractMatrix{<:AbstractFloat}, 
    ky :: AbstractMatrix{<:AbstractFloat}, 
    kz :: AbstractMatrix{<:AbstractFloat}, 
    ki :: AbstractVector{<:AbstractFloat}, 
    ef_bins :: AbstractVector{<:AbstractFloat}, 
    vertices :: AbstractVector{<:Point{3}}, 
    indices :: AbstractVector, 
    e2s :: AbstractVector{<:Point{3}}, 
    e3s :: AbstractVector{<:Point{3}}, 
    mc_coords :: AbstractMatrix{<:AbstractFloat}, 
    len_i :: AbstractVector{<:AbstractFloat}
    )
    A_grid = zeros(n_bins, n_detectors)
    # Determining the locations in which the signal is either NaN or 0 as we don't want to calculate atten there.
    idx = findall(.~((data .== 0) .| (isnan.(data))))
    for I in idx
        kf = [kx[I], ky[I], kz[I]]
        A_grid[I] = atten_calc(ki, SVector{3}(kf), ef_bins[I[1]], vertices, indices, e2s, e3s, mc_coords, len_i)
    end
    return A_grid
end

@benchmark a_grid_calc(data, kx, ky, kz, ki, ef_bins, vertices, indices, e2s, e3s, mc_coords, len_i)

BenchmarkTools.Trial: 1 sample with 1 evaluation per sample.
 Single result which took 10.117 s (17.33% GC) to evaluate,
 with a memory estimate of 43.70 GiB, over 6992688 allocations.

In [105]:
# Testing the time taken to output this grid of attenuation factors.

A_grid = a_grid_calc(data, kx, ky, kz, ki, ef_bins, vertices, indices, e2s, e3s, mc_coords, len_i)
display(A_grid)
display(A_grid[160,6])

320×98304 Matrix{Float64}:
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  …  0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  …  0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 ⋮                        ⋮              ⋱                 ⋮              
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0 

1.0463744196042372e-6

In [ ]:
# Defining the function that calculates the grid of attenuation factors.
# Sparse arrays.

"""
Calculates the attenuation factor for every non-zero, non-NaN, signal and stores in a grid of detector against energy bin.

Parameters
----------
data (n_bins x n_detectors matrix with float elements): Neutron signal measured at different detectors for different energy bins.
kx (n_bins x n_detectors matrix with float elements): Post-scattering neutron wavevector component in x direction, in Angstrom^-1.
ky (n_bins x n_detectors matrix with float elements): Post-scattering neutron wavevector component in y direction, in Angstrom^-1.
kz (n_bins x n_detectors matrix with float elements): Post-scattering neutron wavevector component in z direction, in Angstrom^-1.
ki (3-vector with float elements): Pre-scattering neutron wavevector, in Angstrom^-1.
en_i (float): Pre-scattering neutron energy, in meV.
ef_bins (n_bins-vector with float elements): Post-scattering neutron energy of each bin, in meV.
vertices (n_faces-vector of 3-vectors with float elements): Vertices of the triangular faces.
indices (n_faces-vector of 3-vectors with integer elements): Indices describing which vertices correspond to which triangles.
e2s (n_faces-vector of 3-vectors with float elements): Array containing vectors parallel to each face, equal to V2 - V1.
e3s (n_faces-vector of 3-vectors with float elements): Array containing vectors parallel to each face, equal to V3 - V1.
mc_coords (n_mc-vector of 3-vectors with float elements): Coordinates of sample points used in MC method.
len_i (n_mc-vector with float elements): Pre-scattering path length of neutron, in units of .stl file.

Returns
-------
A_grid (n_bins x n_detectors matrix with float elements): Attenuation factor grid.
"""
function a_grid_calc(
    sdata :: SparseMatrixCSC{Float64, Int}, 
    kx :: AbstractMatrix{Float64}, 
    ky :: AbstractMatrix{Float64}, 
    kz :: AbstractMatrix{Float64}, 
    ki :: SVector{3, Float64}, 
    ef_bins :: SVector{n_bins, Float64}, 
    vertices :: AbstractVector{<:Point{3}}, 
    indices :: AbstractVector, 
    e2s :: AbstractVector{<:Point{3}}, 
    e3s :: AbstractVector{<:Point{3}}, 
    mc_coords :: AbstractMatrix{Float64}, 
    len_i :: AbstractVector{Float64}
    ) :: SparseMatrixCSC{Float64, Int}
    A_grid = spzeros(n_bins, n_detectors)
    # Iterating through the locations in which the signal is not 0 and not NaN.
    # ... splits the tuple so it can be zipped together.
    idx = zip(findnz(sdata)[1:2]...)
    for I in idx
        kf = [kx[I...], ky[I...], kz[I...]]
        A_grid[I...] = atten_calc(ki, SVector{3}(kf), ef_bins[I[1]], vertices, indices, e2s, e3s, mc_coords, len_i)
    end
    return A_grid
end

@benchmark a_grid_calc(sdata, kx, ky, kz, ki, ef_bins, vertices, indices, e2s, e3s, mc_coords, len_i)


BenchmarkTools.Trial: 1 sample with 1 evaluation per sample.
 Single result which took 13.504 s (13.00% GC) to evaluate,
 with a memory estimate of 43.48 GiB, over 6992735 allocations.

In [107]:
# Replacing all the NaNs with 0 to allow for sparse matrix conversion.
# The original data can be re-extracted from the file is needed again.

data .= ifelse.(isnan.(data), 0, data)
sdata = sparse(data)

# Testing the time taken to output this grid of attenuation factors.

A_grid = a_grid_calc(sdata, kx, ky, kz, ki, ef_bins, vertices, indices, e2s, e3s, mc_coords, len_i)
display(A_grid)
display(A_grid[160,6])

320×98304 SparseMatrixCSC{Float64, Int64} with 317849 stored entries:
⎡⣷⣤⣇⣾⣮⣴⣿⣿⣼⣦⣗⣷⣷⣷⣴⣧⣷⣴⣦⣼⣆⣛⣽⣖⣺⣶⣀⣔⣰⣂⣠⣶⣕⣶⣾⣧⣱⣔⣷⣖⎤
⎣⣿⢉⣛⢿⣿⣿⣿⣿⣿⣻⣿⣿⣿⣿⡿⣿⡻⣿⣿⣿⣿⣿⣿⣿⢿⣿⣿⣿⣿⣿⣿⣿⣿⢿⡿⣿⢿⣿⣿⣿⎦

1.0463744196042372e-6